# 3. Properties (Getters/Setters)

In Java, you write getters and setters from day one because switching from a field to a method is a **breaking change**. Python's `@property` solves this beautifully — start with a plain attribute, and add validation later **without changing any client code**.

This notebook covers: `@property`, `@setter`, `@deleter`, computed properties, `@cached_property`, and the Pythonic philosophy of attribute access.

### 3.1 Why Properties? The Java Problem

**☕ JAVA:** You must write getters/setters upfront, because changing `person.age` to `person.getAge()` later breaks all client code:
```java
// Start with a field:
public int age;           // client: person.age = 5

// Later you need validation — BREAKING CHANGE:
private int age;
public int getAge() { return age; }
public void setAge(int age) {
    if (age < 0) throw new IllegalArgumentException();
    this.age = age;         // client must change to: person.setAge(5)
}
```

**🐍 PYTHON:** No problem! Start with a plain attribute. If you need validation later, add `@property` — **client code stays exactly the same** (`person.age = 5` still works).

In [1]:
class PersonV1:
    def __init__(self, name: str, age: str):
        self.name = name
        self.age = age

pv1 = PersonV1("Alice", 30)
print(pv1.name, ":" , pv1.age )

Alice : 30


In [4]:
class PersonV2:
    def __init__(self, name: str, age: str):
        self.__name = name
        self.__age = age

    def set_age(self, age):
        if age > 0:
            self.__age = age
        self.__age = 1

    def get_age(self):
        return self.__age

pv2 = PersonV2("Alice", 30)
pv2.set_age(-100)
print(f"Age: {pv2.get_age()}")

Age: 1


#### Because this above don't is "Pythonial" must work like this below.

In [7]:
class PersonV3:
    def __init__(self, name: str, age: str):
        self.__name = name
        self.__age = age

    def set_age(self, age):
        if age > 0:
            self.__age = age
        else:
            self.__age = 1

    def get_age(self):
        return self.__age
    
    age = property(get_age, set_age)

p = PersonV3("Alice", 30)
print(f"Age: {p.age}")
p.age = 40
print(f"Age: {p.age}")

# pv3 = PersonV3("Alice", 30)
# pv3.set_age(-100)
# print(f"Age: {pv3.get_age()}")

Age: 30
Age: 40


### 3.2 Adding a Getter with `@property`

**☕ JAVA:** `public int getAge() { return this.age; }`

**🐍 PYTHON:** `@property` turns a method into an attribute-like access. Store the real value in a `_private` variable.

In [10]:
class User:
    def __init__(self, name: str, age: str):
        self.name = name
        self.__age = age

    @property
    def age(self) -> int:
        return self.__age
    
    @age.setter
    def age(self, value: int):
        if value < 0:
            raise ValueError(f"Age cannot be less than 0. Got: {value}")
        if value > 150:
            raise ValueError(f"Age unrealistic, got {value}")
    
        self.__age = value

user = User("Alice", 35)
print(f"Age: {user.age}") # getter
user.age = 40 # setter
print(f"Updated Age: {user.age}")

Age: 35
Updated Age: 40


### 3.3 Adding a Setter with Validation

**☕ JAVA:**
```java
public void setAge(int age) {
    if (age < 0) throw new IllegalArgumentException("Age cannot be negative");
    this.age = age;
}
```

**🐍 PYTHON:** Use `@property_name.setter`. The client still writes `user.age = 31` — same syntax as a plain attribute!

In [13]:
class PersonVal: 
    def __init__(self, name: str, age: str):
        self.name = name
        self.__age = age

    @property
    def age(self):
        return self.__age
    
    @age.setter
    def age(self, value: int):
        # I can apply some validations
        if value < 0:
            raise ValueError(f"Age cannot be negative. I got: {value}")
        self.age = value

bob = PersonVal("Bob", 45)
print(f"{bob.name} is {bob.age}")

Bob is 45


> 💡 **Key insight:** Notice that `self.age = age` in `__init__` also goes through the setter — giving you validation at construction time too!

### 3.4 Computed (Read-Only) Properties

**☕ JAVA:** Computed values are always methods: `rectangle.getArea()`

**🐍 PYTHON:** A `@property` without a setter is automatically **read-only**. Great for values derived from other attributes.

In [14]:
class Rectangle:
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height
    
    @property
    def area(self):
        return self.width * self.height
    
    @property
    def perimeter(self):
        return 2 * (self.width + self.height)
    
    @property
    def is_square(self):
        return self.width == self.height
    
rect = Rectangle(10, 20)
print(f"Area: {rect.area}")
print(f"Perimeter: {rect.perimeter}")
print(f"Is square?: {rect.is_square}")

Area: 200
Perimeter: 60
Is square?: False


### 3.5 The `@deleter` — Optional Cleanup

**☕ JAVA:** No equivalent — fields can't be "deleted".

**🐍 PYTHON:** `@property_name.deleter` lets you hook into `del obj.attr`. Rarely used, but useful for cleanup or resetting state.

In [16]:
class CachedProfile:
    def __init__(self, username: str):
        self.username = username
        self._avatar = None
    
    @property
    def avatar(self) -> str:
        if self._avatar is None:
            self._avatar = f"avatar_{self.username}.png"
            print("(generated avatar)") 
        return self._avatar
    
    @avatar.deleter
    def avatar(self):
        print("(cache clear)")
        self._avatar = None

profile = CachedProfile("panos")
print(f"Avatar: {profile.avatar}")
print(f"Avatar: {profile.avatar}")

del profile.avatar
print(f"Avatar: {profile.avatar}")

(generated avatar)
Avatar: avatar_panos.png
Avatar: avatar_panos.png
(cache clear)
(generated avatar)
Avatar: avatar_panos.png


### 3.6 `@cached_property` — Built-in Lazy Caching (Python 3.8+)

The manual caching pattern in 3.5 is so common that Python provides `functools.cached_property`. It computes the value **once**, then stores it as a regular attribute — no repeated computation.

| Feature | `@property` | `@cached_property` |
|---------|------------|--------------------|
| Recalculated on each access | ✅ Yes | ❌ No — computed once |
| Supports setter | ✅ Yes | ❌ No |
| Thread-safe | N/A | ✅ Yes (Python 3.12+) |
| Clear cache | Via `@deleter` | `del obj.attr` |

In [ ]:
from functools import cached_property
import time

class DataAnnalysis:
    def __init__(self, data: list[int]):
        self.data = data

    @cached_property
    def summary(self) -> dict:
        print("(computing summury)...")
        time.sleep(1)
        return {
            "mean" : sum(self.data)/ len(self.data),
            "min" : min(self.data),
            "max" : max(self.data),
            "count" : len(self.data)
        }

analysis = DataAnnalysis([10, 20, 30, 40, 50])
print(f"First access: {analysis.summary}")
print(f"Second access: {analysis.summary}")

del analysis.summary
print(f"After deletion: {analysis.summary}")

(computing summury)...
First access: {'mean': 30.0, 'min': 10, 'max': 50, 'count': 5}
Second access: {'mean': 30.0, 'min': 10, 'max': 50, 'count': 5}
(computing summury)...
After deletion: {'mean': 30.0, 'min': 10, 'max': 50, 'count': 5}


> ⚠️ **Important:** `@cached_property` only works on instances that have a `__dict__` (i.e., no `__slots__`). It stores the result directly in the instance dictionary.

### 3.7 How Properties Work — The Descriptor Protocol

**☕ JAVA:** Annotations like `@Override` are compile-time metadata — they don't change how attribute access works.

**🐍 PYTHON:** `@property` is actually a **descriptor** — a class that implements `__get__`, `__set__`, and `__delete__`. When Python sees `obj.age`, it checks if `age` in the class is a descriptor and calls its methods. This is not magic — it's a protocol you can implement yourself!

In [24]:
# Product: price, quantity

class Positive:
    """A positive descriptor that enforce positive values."""
    def __init__(self, name: str):
        self.name = name
        self.private_name = f"_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None: 
            return self
        return getattr(obj, self.private_name)
    
    def __set__(self, obj, value):
        if value < 0:
            raise ValueError(f"{self.name} must be positive. I got: {value}")
        setattr( obj, self.private_name, value)


class Product:
    price = Positive("price")
    quantity = Positive("quantity")

    def __init__(self, name: str, price: float, quantity: int):
        self.name = name
        self.price = price          # Positive.__set__
        self.quantity = quantity    # Positive.__set__

p1 = Product("I-Phone", 999.99, 100)
print(f"{p1.name}: ${p1.price} x {p1.quantity}")

try:
    p1.price = -150
except ValueError as e:
    print(f"Descriptor: {e}")

print(f"{p1.name}: ${p1.price} x {p1.quantity}")

I-Phone: $999.99 x 100
Descriptor: price must be positive. I got: -150
I-Phone: $999.99 x 100


### Κάποια επιπλέον πραγματάκια:
##### Αυτά τα δύο κάνουν ΑΚΡΙΒΩΣ το ίδιο πράγμα:
```Python
p.name            # hardcoded — ξέρεις το όνομα εκ των προτέρων
getattr(p, "name")  # δυναμικό — το όνομα είναι string, μπορεί να αλλάζει
```
 
##### Και αντίστοιχα για εγγραφή:
```Python
p.name = "Widget"
setattr(p, "name", "Widget")
```

```Python
# Βήμα 1: Δημιουργία descriptor
price = Positive("price")
# Τώρα μέσα στον descriptor:
#   self.name = "price"
#   self.private_name = "_price"
 
# Βήμα 2: self.price = 9.99 καλεί τη __set__
def __set__(self, obj, value):          # obj = το Product instance, value = 9.99
    if value <= 0:
        raise ValueError(...)
    setattr(obj, self.private_name, value)
    # ↓ ισοδυναμεί με ↓
    # obj._price = 9.99
    # Αλλά δεν μπορείς να γράψεις obj._price γιατί
    # το "_price" είναι string που υπολογίστηκε δυναμικά!
 
# Βήμα 3: print(p.price) καλεί τη __get__
def __get__(self, obj, objtype=None):   # obj = το Product instance
    return getattr(obj, self.private_name)
    # ↓ ισοδυναμεί με ↓
    # return obj._price

```

> 💡 **When to use descriptors over `@property`:** When you have the **same validation logic** on multiple attributes. Instead of writing 3 identical properties, write one descriptor and reuse it.

### 3.8 Write-Only Properties — A Rare Pattern

Python allows a setter without a meaningful getter — useful for security-sensitive values like passwords where reading back the raw value should be blocked.

In [30]:
import hashlib

class Account:
    def __init__(self, username: str, password: str):
        self.username = username
        self.password = password

    @property
    def password(self) -> str:
            raise AttributeError("Password is write-only")
    
    @password.setter
    def password(self, value: str) -> None:
         if len(value) < 8:
              raise ValueError("Password must be at least 8 characters.")
         self._password_hash = hashlib.sha256(value.encode()).hexdigest()
    
    def verify(self, password: str) -> bool:
         return self._password_hash == hashlib.sha256(password.encode()).hexdigest()

acc1 = Account("alice", "123456798")
print(f"Verify correct: {acc1.verify("123456798")}")
print(f"Verify wrong: {acc1.verify("123454798")}")

try:
    print(acc1.password)
except AttributeError as e:
    print(f"Error: {e}")

Verify correct: True
Verify wrong: False
Error: Password is write-only


---

## 🧪 Try It Yourself

**Exercise 1:** Create a `Temperature` class with a `celsius` property (getter/setter). Add a read-only `fahrenheit` property computed as `C × 9/5 + 32`. The setter should reject values below -273.15 (absolute zero).

In [ ]:
# Exercise 1: Your code here


**Exercise 2:** Create a `Password` class with a `password` property. The setter should:
- Reject passwords shorter than 8 characters
- Store the password as a hash (use `hash()` for simplicity)
- The getter should return `"****"` (never reveal the actual password)

In [ ]:
# Exercise 2: Your code here


**Exercise 3:** Create a `Circle` class with a `radius` property (rejects negative values). Add computed read-only properties for `area` and `circumference`. Use `@cached_property` for `area` and show how modifying `radius` requires invalidating the cache.

In [ ]:
# Exercise 3: Your code here


---

## 📝 Key Takeaways: Java → Python

| Concept | Java | Python |
|---------|------|--------|
| Getter | `getAge()` | `@property` |
| Setter | `setAge(value)` | `@age.setter` |
| Deleter | Not possible | `@age.deleter` |
| Client syntax | `obj.getAge()` / `obj.setAge(5)` | `obj.age` / `obj.age = 5` |
| Read-only | `private` field + getter only | `@property` without setter |
| Write-only | Not common | Getter raises `AttributeError` |
| Computed value | `getArea()` method | `@property` — looks like an attribute |
| Lazy caching | Manual field + null check | `@cached_property` (Python 3.8+) |
| Add validation later | Breaks all client code | Seamless — no changes needed |
| Backing field | `private int age;` | `self._age` (convention) |
| Reusable validation | Custom annotation processor | Descriptor protocol |
| Best practice | Always write getters/setters | Start with plain attributes |